# EU AI Act & UK AI Governance Compliance Checker
## 7CS525 – Human and Legal Aspects of Cyber Security | Assessment 01
### Automation Artefact — Component 5

**System under analysis:** AI-Enabled Biometric Facial Recognition Access Control  
**Context:** UK Central Government (Ministry-level secure estate)  

---

### Purpose of this Notebook
This automation artefact operationalises the legal and regulatory analysis conducted in Components 1–3 of the portfolio.  
It provides a reproducible, structured workflow that:

1. Classifies an AI system against the **EU AI Act** risk tiers (Annex III)
2. Maps **applicable legal obligations** to each identified risk dimension
3. Generates a colour-coded **AI Risk & Compliance Register** (pandas DataFrame)
4. Produces a **risk profile visualisation** (matplotlib)
5. Exports a **structured compliance checklist** to CSV

**Relevance to governance practice:** This notebook demonstrates how automation can systematise regulatory mapping tasks that would otherwise be manual, inconsistent, and difficult to audit — directly supporting the kind of governance workflows mandated by EU AI Act Art. 9 (Risk Management System) and UK GDPR Art. 35 (DPIA).

---

In [ ]:
# ── Cell 1: Imports ─────────────────────────────────────────────────────────
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_rows', 20)
print(f'Notebook initialised: {datetime.now().strftime("%Y-%m-%d %H:%M")}')
print('EU AI Act & UK Governance Compliance Checker — 7CS525 Assessment 01')

---
## Part 1: System Profile Definition

The system profile is the structured input that drives all subsequent classification and compliance mapping.  
Modify these parameters to analyse a different AI system.

In [ ]:
# ── Cell 2: System Profile ───────────────────────────────────────────────────
# Define the AI system under analysis
# Change these parameters to analyse any AI-enabled cyber security system

system_profile = {
    # ── Identity
    "system_name":                   "FacePass Gov v2.1",
    "deploying_organisation":        "UK Home Office – Estate Management",
    "developer":                     "NEC Corporation (hypothetical deployment)",
    "deployment_context":            "UK Central Government secure buildings",
    "analysis_date":                 datetime.now().strftime("%Y-%m-%d"),
    
    # ── Technical characteristics
    "processes_biometrics":          True,   # facial, voice, fingerprint, etc.
    "real_time_biometric_id":        True,   # live identification (not verification)
    "uses_machine_learning":         True,
    "automated_decision_making":     True,   # grants/denies access without human approval
    "human_override_available":      True,   # security officer can override
    "hitl_routine":                  False,  # HITL only at exception node, not routine
    "explainability_available":      False,  # no XAI dashboard
    "logging_implemented":           True,
    
    # ── Scope of impact
    "affects_govt_building_access":  True,
    "affects_critical_infrastructure": True,
    "subjects_count":                14000,  # staff + daily visitors
    "processes_children":            False,
    "cross_border_data_transfer":    False,
    
    # ── Legal/compliance posture
    "dpia_completed":                False,  # Data Protection Impact Assessment
    "bias_audit_completed":          False,
    "demographic_accuracy_tested":   False,
    "conformity_assessment_done":    False,  # EU AI Act requirement for High Risk
    "ico_consultation_completed":    False,
    "equality_impact_assessment":    False,  # Equality Act 2010 / PSED
    "staff_training_provided":       False,
    
    # ── Data handling
    "biometric_data_encrypted":      True,
    "data_retention_policy_exists":  True,
    "purpose_limitation_documented": True,
    "consent_or_lawful_basis":       "Legitimate interests (Article 9(2)(b) DPA 2018)",
}

print(f"System Profile Loaded: {system_profile['system_name']}")
print(f"Deployer: {system_profile['deploying_organisation']}")
print(f"Analysis Date: {system_profile['analysis_date']}")

---
## Part 2: EU AI Act Risk Classification

The EU AI Act (Regulation 2024/1689) establishes four risk categories.  
This function implements the classification logic based on Art. 5 (Unacceptable), Art. 6 + Annex III (High Risk), and Arts. 50/69 (Limited/Minimal).

In [ ]:
# ── Cell 3: EU AI Act Classification Engine ──────────────────────────────────

def classify_eu_ai_act(profile: dict) -> dict:
    """
    Classify an AI system under the EU AI Act risk framework.
    
    Returns a dict with:
        - risk_level: str
        - justification: list[str]
        - annex_refs: list[str]
        - prohibited: bool
    """
    justifications = []
    annex_refs = []
    
    # ── Tier 1: Unacceptable Risk (Art. 5) ──────────────────────────────────
    # Real-time remote biometric ID in publicly accessible spaces for law enforcement
    # NOTE: Government building access is NOT a publicly accessible space under Art.5
    # so this system does not meet the Art.5 prohibition threshold
    prohibited = False
    
    # ── Tier 2: High Risk (Art. 6 + Annex III) ──────────────────────────────
    high_risk_triggers = []
    
    if profile.get("processes_biometrics") and profile.get("real_time_biometric_id"):
        high_risk_triggers.append(
            "Biometric identification system — Annex III §1(a): AI systems intended for "
            "biometric identification of natural persons"
        )
        annex_refs.append("Annex III, Category 1(a)")
    
    if profile.get("affects_critical_infrastructure"):
        high_risk_triggers.append(
            "Critical infrastructure — Annex III §2: AI systems used as safety components "
            "in critical infrastructure management"
        )
        annex_refs.append("Annex III, Category 2")
    
    if profile.get("affects_govt_building_access"):
        high_risk_triggers.append(
            "Access control to essential services — Annex III §5(b): AI systems intended "
            "to be used for evaluation of persons for access to essential public services"
        )
        annex_refs.append("Annex III, Category 5(b)")
    
    if profile.get("automated_decision_making") and profile.get("subjects_count", 0) > 1000:
        high_risk_triggers.append(
            "Large-scale automated decision making affecting significant number of "
            f"individuals ({profile['subjects_count']:,} subjects)"
        )
    
    justifications.extend(high_risk_triggers)
    
    if prohibited:
        risk_level = "UNACCEPTABLE"
    elif len(high_risk_triggers) >= 1:
        risk_level = "HIGH RISK"
    elif profile.get("automated_decision_making"):
        risk_level = "LIMITED RISK"
        justifications.append("Automated decision-making with transparency obligations (Art. 50)")
    else:
        risk_level = "MINIMAL RISK"
        justifications.append("No triggering conditions for higher risk categories identified")
    
    return {
        "risk_level":     risk_level,
        "justification":  justifications,
        "annex_refs":     list(set(annex_refs)),
        "prohibited":     prohibited,
        "trigger_count":  len(high_risk_triggers),
    }


# Run classification
classification = classify_eu_ai_act(system_profile)

# Display result
risk_colours = {
    "UNACCEPTABLE": "\033[91m",   # red
    "HIGH RISK":    "\033[93m",   # yellow
    "LIMITED RISK": "\033[94m",   # blue
    "MINIMAL RISK": "\033[92m",   # green
}
RESET = "\033[0m"
col = risk_colours.get(classification['risk_level'], '')

print("=" * 70)
print(f"EU AI ACT CLASSIFICATION: {col}{classification['risk_level']}{RESET}")
print("=" * 70)
print(f"Triggering conditions ({classification['trigger_count']} identified):")
for i, j in enumerate(classification['justification'], 1):
    print(f"  {i}. {j}")
print(f"\nAnnex III references: {', '.join(classification['annex_refs'])}")

---
## Part 3: Legal Obligations Mapping

For each risk level, this function maps the mandatory legal obligations across the EU AI Act, UK GDPR, and other applicable instruments.

In [ ]:
# ── Cell 4: Legal Obligations Mapper ─────────────────────────────────────────

def get_legal_obligations(risk_level: str, profile: dict) -> list[dict]:
    """
    Returns a list of applicable legal obligations given the risk classification
    and system profile. Covers EU AI Act, UK GDPR, Equality Act, HRA 1998.
    """
    obligations = []
    
    # ── EU AI Act obligations for HIGH RISK systems ──────────────────────────
    if risk_level in ["HIGH RISK", "UNACCEPTABLE"]:
        eu_aia_obligations = [
            {"instrument": "EU AI Act", "article": "Art. 9",
             "obligation": "Risk Management System",
             "detail": "Continuous identification, analysis and mitigation of risks throughout lifecycle",
             "compliance_status": "NOT MET" if not profile.get("bias_audit_completed") else "MET"},
            {"instrument": "EU AI Act", "article": "Art. 10",
             "obligation": "Data Governance & Quality",
             "detail": "Training data must be representative; bias testing across demographic groups required",
             "compliance_status": "NOT MET" if not profile.get("demographic_accuracy_tested") else "MET"},
            {"instrument": "EU AI Act", "article": "Art. 11",
             "obligation": "Technical Documentation",
             "detail": "Full technical documentation before market placement; maintained and updated",
             "compliance_status": "UNKNOWN"},
            {"instrument": "EU AI Act", "article": "Art. 12",
             "obligation": "Record-Keeping & Logging",
             "detail": "Automatic event logging sufficient to identify risks post-deployment",
             "compliance_status": "MET" if profile.get("logging_implemented") else "NOT MET"},
            {"instrument": "EU AI Act", "article": "Art. 13",
             "obligation": "Transparency to Deployers",
             "detail": "Instructions for use; capabilities and limitations disclosed to deployer",
             "compliance_status": "UNKNOWN"},
            {"instrument": "EU AI Act", "article": "Art. 14",
             "obligation": "Human Oversight",
             "detail": "System must allow operators to monitor, intervene, override, or halt",
             "compliance_status": "PARTIAL" if profile.get("human_override_available") and not profile.get("hitl_routine") else
                                  "MET" if profile.get("hitl_routine") else "NOT MET"},
            {"instrument": "EU AI Act", "article": "Art. 15",
             "obligation": "Accuracy, Robustness & Cybersecurity",
             "detail": "Declared accuracy levels; performance across demographic groups; adversarial testing",
             "compliance_status": "NOT MET" if not profile.get("demographic_accuracy_tested") else "MET"},
            {"instrument": "EU AI Act", "article": "Art. 16 + 26",
             "obligation": "Developer & Deployer Obligations",
             "detail": "Conformity assessment; registration in EU AI database; post-market monitoring",
             "compliance_status": "NOT MET" if not profile.get("conformity_assessment_done") else "MET"},
            {"instrument": "EU AI Act", "article": "Art. 86",
             "obligation": "Right to Explanation",
             "detail": "Affected persons have right to explanation of AI-assisted individual decisions",
             "compliance_status": "NOT MET" if not profile.get("explainability_available") else "MET"},
        ]
        obligations.extend(eu_aia_obligations)
    
    # ── UK GDPR / DPA 2018 obligations ──────────────────────────────────────
    if profile.get("processes_biometrics"):
        gdpr_obligations = [
            {"instrument": "UK GDPR / DPA 2018", "article": "Art. 9",
             "obligation": "Special Category Data Processing",
             "detail": "Lawful basis required for biometric processing; explicit consent or statutory gateway",
             "compliance_status": "MET" if profile.get("consent_or_lawful_basis") else "NOT MET"},
            {"instrument": "UK GDPR / DPA 2018", "article": "Art. 35",
             "obligation": "Data Protection Impact Assessment (DPIA)",
             "detail": "Mandatory DPIA for biometric data processing at scale",
             "compliance_status": "MET" if profile.get("dpia_completed") else "NOT MET"},
            {"instrument": "UK GDPR / DPA 2018", "article": "Art. 22",
             "obligation": "Automated Decision-Making Rights",
             "detail": "Right not to be subject to solely automated decisions with significant effects",
             "compliance_status": "PARTIAL" if profile.get("human_override_available") else "NOT MET"},
            {"instrument": "UK GDPR / DPA 2018", "article": "Arts. 33–34",
             "obligation": "Breach Notification",
             "detail": "72-hour notification to ICO; notification to data subjects if high risk",
             "compliance_status": "UNKNOWN"},
        ]
        obligations.extend(gdpr_obligations)
    
    # ── Equality Act 2010 obligations ────────────────────────────────────────
    obligations.append({
        "instrument": "Equality Act 2010",
        "article": "s.19 + s.149 (PSED)",
        "obligation": "Indirect Discrimination & Public Sector Equality Duty",
        "detail": "Biometric accuracy disparities across protected characteristics may constitute indirect discrimination",
        "compliance_status": "NOT MET" if not profile.get("equality_impact_assessment") else "MET"
    })
    
    # ── Human Rights Act 1998 ────────────────────────────────────────────────
    obligations.append({
        "instrument": "Human Rights Act 1998",
        "article": "Art. 8 ECHR",
        "obligation": "Right to Private Life — Proportionality",
        "detail": "Biometric processing must be necessary, proportionate and pursue a legitimate aim (Bridges [2020])",
        "compliance_status": "UNKNOWN"
    })
    
    # ── ICO Consultation ─────────────────────────────────────────────────────
    if not profile.get("ico_consultation_completed") and risk_level == "HIGH RISK":
        obligations.append({
            "instrument": "UK GDPR / DPA 2018",
            "article": "Art. 36",
            "obligation": "Prior Consultation with ICO",
            "detail": "Where DPIA indicates high residual risk, prior consultation with ICO required",
            "compliance_status": "NOT MET"
        })
    
    return obligations


# Generate obligations
obligations = get_legal_obligations(classification['risk_level'], system_profile)
df_obligations = pd.DataFrame(obligations)

print(f"Total obligations identified: {len(obligations)}")
met     = sum(1 for o in obligations if o['compliance_status'] == 'MET')
partial = sum(1 for o in obligations if o['compliance_status'] == 'PARTIAL')
not_met = sum(1 for o in obligations if o['compliance_status'] == 'NOT MET')
unknown = sum(1 for o in obligations if o['compliance_status'] == 'UNKNOWN')
print(f"  ✅ MET:      {met}")
print(f"  ⚠️  PARTIAL:  {partial}")
print(f"  ❌ NOT MET:  {not_met}")
print(f"  ❓ UNKNOWN:  {unknown}")

---
## Part 4: Colour-Coded Compliance Register

The register below presents all obligations with colour-coded compliance status.  
This mirrors the format of a professional governance register used in real public sector deployments.

In [ ]:
# ── Cell 5: Styled Compliance Register ──────────────────────────────────────

def colour_compliance(val):
    """Apply background colour to compliance status cells."""
    colour_map = {
        'MET':      'background-color: #d4edda; color: #155724; font-weight: bold',
        'PARTIAL':  'background-color: #fff3cd; color: #856404; font-weight: bold',
        'NOT MET':  'background-color: #f8d7da; color: #721c24; font-weight: bold',
        'UNKNOWN':  'background-color: #d1ecf1; color: #0c5460; font-style: italic',
    }
    return colour_map.get(val, '')

def colour_instrument(val):
    """Colour-code by regulatory instrument."""
    if 'EU AI Act' in val:
        return 'background-color: #e8eeff; color: #1a275e; font-weight: bold'
    elif 'GDPR' in val or 'DPA' in val:
        return 'background-color: #e8f4f8; color: #0c3547'
    elif 'Equality' in val:
        return 'background-color: #f0f8e8; color: #2d5a1b'
    elif 'Human Rights' in val:
        return 'background-color: #fef9e8; color: #5a3e1b'
    return ''

# Style the DataFrame
styled = (
    df_obligations
    .style
    .applymap(colour_compliance, subset=['compliance_status'])
    .applymap(colour_instrument, subset=['instrument'])
    .set_caption(
        f"AI Legal Obligations Register — {system_profile['system_name']} "
        f"| Classification: {classification['risk_level']} "
        f"| Generated: {system_profile['analysis_date']}"
    )
    .set_properties(**{'font-size': '12px', 'border': '1px solid #dee2e6'})
    .set_table_styles([
        {'selector': 'caption', 'props': 'font-weight: bold; font-size: 14px; margin-bottom: 8px; color: #1a275e;'},
        {'selector': 'th', 'props': 'background-color: #1a275e; color: white; padding: 8px; text-align: left;'},
        {'selector': 'td', 'props': 'padding: 6px 8px; vertical-align: top;'},
    ])
)

styled

---
## Part 5: Risk Profile Visualisation

The chart below visualises the compliance posture across all identified obligations,  
broken down by regulatory instrument — demonstrating the multi-framework nature of AI governance.

In [ ]:
# ── Cell 6: Risk Profile Visualisation ──────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.patch.set_facecolor('#f5f7fd')

STATUS_COLOURS = {
    'MET':     '#16a34a',
    'PARTIAL': '#f0b323',
    'NOT MET': '#dc2626',
    'UNKNOWN': '#0369a1',
}

# ── Chart 1: Overall compliance donut ────────────────────────────────────────
ax1 = axes[0]
ax1.set_facecolor('#f5f7fd')
status_counts = df_obligations['compliance_status'].value_counts()

# Ensure consistent order
order = ['MET', 'PARTIAL', 'NOT MET', 'UNKNOWN']
sizes  = [status_counts.get(s, 0) for s in order]
colours = [STATUS_COLOURS[s] for s in order]
labels = [f"{s}\n({status_counts.get(s, 0)})" for s in order]

wedges, texts, autotexts = ax1.pie(
    sizes, labels=labels, colors=colours,
    autopct='%1.0f%%', startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2},
    pctdistance=0.75,
    textprops={'fontsize': 11},
    counterclock=False,
)
for at in autotexts:
    at.set_fontsize(10)
    at.set_color('white')
    at.set_fontweight('bold')

# Donut hole
centre_circle = plt.Circle((0,0), 0.55, fc='#f5f7fd')
ax1.add_artist(centre_circle)
ax1.text(0, 0.08, f"{len(obligations)}", ha='center', va='center',
         fontsize=26, fontweight='bold', color='#1a275e')
ax1.text(0, -0.22, "obligations", ha='center', va='center',
         fontsize=10, color='#64748b')
ax1.set_title('Overall Compliance Posture', fontsize=14, fontweight='bold',
               color='#1a275e', pad=15)


# ── Chart 2: Obligations by instrument & status (stacked bar) ────────────────
ax2 = axes[1]
ax2.set_facecolor('#f5f7fd')

# Group by instrument
pivot = df_obligations.pivot_table(
    index='instrument', columns='compliance_status',
    aggfunc='size', fill_value=0
)
# Ensure all columns present
for s in ['MET', 'PARTIAL', 'NOT MET', 'UNKNOWN']:
    if s not in pivot.columns:
        pivot[s] = 0
pivot = pivot[['MET', 'PARTIAL', 'NOT MET', 'UNKNOWN']]

# Shorten instrument names for display
short_names = {
    'EU AI Act': 'EU AI Act',
    'UK GDPR / DPA 2018': 'UK GDPR\n/ DPA 2018',
    'Equality Act 2010': 'Equality\nAct 2010',
    'Human Rights Act 1998': 'HRA 1998',
}
pivot.index = [short_names.get(i, i) for i in pivot.index]

bottom = np.zeros(len(pivot))
for status in ['MET', 'PARTIAL', 'NOT MET', 'UNKNOWN']:
    vals = pivot[status].values
    bars = ax2.barh(pivot.index, vals, left=bottom,
                    color=STATUS_COLOURS[status], edgecolor='white', linewidth=1.5,
                    label=status, height=0.55)
    # Add value labels inside bars
    for bar, val in zip(bars, vals):
        if val > 0:
            x = bar.get_x() + bar.get_width() / 2
            y = bar.get_y() + bar.get_height() / 2
            ax2.text(x, y, str(int(val)), ha='center', va='center',
                     fontsize=11, fontweight='bold', color='white')
    bottom += vals

ax2.set_xlabel('Number of Obligations', fontsize=11, color='#64748b')
ax2.set_title('Obligations by Instrument & Status', fontsize=14, fontweight='bold',
               color='#1a275e', pad=15)
ax2.legend(loc='lower right', fontsize=10, framealpha=0.9)
ax2.tick_params(colors='#1a275e', labelsize=10)
ax2.spines[['top', 'right']].set_visible(False)
ax2.grid(axis='x', alpha=0.3, color='#94a3b8')
ax2.set_xlim(0, max(pivot.sum(axis=1)) + 1)


# ── Figure title + annotation ─────────────────────────────────────────────────
fig.suptitle(
    f"AI Compliance Risk Profile — {system_profile['system_name']}\n"
    f"EU AI Act Classification: {classification['risk_level']}  |  "
    f"Context: {system_profile['deployment_context']}",
    fontsize=14, fontweight='bold', color='#1a275e', y=1.02
)
fig.text(0.5, -0.02,
    f"Generated by 7CS525 Compliance Checker  |  Analysis date: {system_profile['analysis_date']}  |  "
    "Sources: EU AI Act (2024/1689); UK GDPR; DPA 2018; Equality Act 2010; HRA 1998",
    ha='center', fontsize=8, color='#94a3b8'
)

plt.tight_layout()
plt.savefig('/home/claude/project/compliance_risk_profile.png',
            dpi=150, bbox_inches='tight', facecolor='#f5f7fd')
plt.show()
print("Chart saved: compliance_risk_profile.png")

---
## Part 6: Demographic Accuracy Gap Visualisation

Drawing on NIST FRVT (2019) and Buolamwini & Gebru (2018), this section visualises  
the documented accuracy disparities that constitute both a technical and legal compliance risk.

In [ ]:
# ── Cell 7: Demographic Bias Visualisation ───────────────────────────────────
# Based on NIST FRVT 2019 (NISTIR 8280) findings — illustrative trends

fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor('#f5f7fd')
ax.set_facecolor('#f5f7fd')

demographic_groups = [
    'White Male', 'White Female',
    'Asian Male', 'Asian Female',
    'Black Male', 'Black Female'
]
# False Positive Rates (illustrative, based on NIST FRVT 2019 trend data)
fpr_values = [0.03, 0.08, 0.11, 0.18, 0.21, 0.38]
# False Negative Rates (illustrative)
fnr_values = [0.04, 0.06, 0.09, 0.13, 0.15, 0.24]

x = np.arange(len(demographic_groups))
width = 0.35

# Colour by risk level
def bar_colour(rate):
    if rate < 0.10: return '#16a34a'
    elif rate < 0.20: return '#f0b323'
    else: return '#dc2626'

fpr_colours = [bar_colour(r) for r in fpr_values]
fnr_colours = [bar_colour(r) for r in fnr_values]

bars1 = ax.bar(x - width/2, [r*100 for r in fpr_values], width,
               color=fpr_colours, alpha=0.85, edgecolor='white', linewidth=1.5,
               label='False Positive Rate (%)')
bars2 = ax.bar(x + width/2, [r*100 for r in fnr_values], width,
               color=fnr_colours, alpha=0.55, edgecolor='white', linewidth=1.5,
               label='False Negative Rate (%)', hatch='//')

# Value labels
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4,
            f'{bar.get_height():.0f}%', ha='center', va='bottom',
            fontsize=9, fontweight='bold', color='#1a275e')
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4,
            f'{bar.get_height():.0f}%', ha='center', va='bottom',
            fontsize=9, color='#64748b')

# Threshold line (acceptable < 5% FPR)
ax.axhline(y=5, color='#dc2626', linestyle='--', linewidth=1.5, alpha=0.7)
ax.text(5.6, 5.3, 'Acceptable FPR threshold (5%)', fontsize=8,
        color='#dc2626', va='bottom')

ax.set_xlabel('Demographic Group', fontsize=12, color='#1a275e')
ax.set_ylabel('Error Rate (%)', fontsize=12, color='#1a275e')
ax.set_title(
    'Illustrative Facial Recognition Error Rates by Demographic Group\n'
    '(Based on NIST FRVT 2019 trends — Buolamwini & Gebru, 2018)',
    fontsize=13, fontweight='bold', color='#1a275e'
)
ax.set_xticks(x)
ax.set_xticklabels(demographic_groups, fontsize=10, color='#1a275e')
ax.legend(fontsize=10, framealpha=0.9)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.3, color='#94a3b8')
ax.set_ylim(0, 45)

# Legal risk annotation
ax.annotate(
    'Equality Act 2010 s.19 risk:\nIndirect discrimination if\nFPR disparity is unjustified',
    xy=(5, 38), xytext=(4.0, 42),
    fontsize=8.5, color='#dc2626',
    bbox=dict(boxstyle='round,pad=0.3', facecolor='#fff0f0', edgecolor='#dc2626'),
    arrowprops=dict(arrowstyle='->', color='#dc2626')
)

fig.text(0.5, -0.03,
    'Sources: NIST FRVT 2019 (NISTIR 8280); Buolamwini & Gebru (2018). '
    'Values are illustrative trend representations, not specific vendor figures.',
    ha='center', fontsize=8, color='#94a3b8'
)

plt.tight_layout()
plt.savefig('/home/claude/project/demographic_bias_chart.png',
            dpi=150, bbox_inches='tight', facecolor='#f5f7fd')
plt.show()
print("Chart saved: demographic_bias_chart.png")

---
## Part 7: Compliance Checklist Export

Exports the full compliance register to CSV — suitable for submission to a Data Protection Officer,  
governance board, or as evidence for a DPIA. This operationalises the governance workflow  
mandated by EU AI Act Art. 9 and UK GDPR Art. 35.

In [ ]:
# ── Cell 8: Export & Summary Report ─────────────────────────────────────────

# Add metadata columns for the export
df_export = df_obligations.copy()
df_export.insert(0, 'system_name', system_profile['system_name'])
df_export.insert(1, 'risk_level', classification['risk_level'])
df_export.insert(2, 'analysis_date', system_profile['analysis_date'])
df_export['action_owner'] = df_export['instrument'].apply(
    lambda x: 'Home Office CISO' if 'EU AI Act' in x else
              'Data Protection Officer' if 'GDPR' in x or 'DPA' in x else
              'Legal Counsel' if 'Equality' in x or 'Human Rights' in x else 'TBC'
)
df_export['priority'] = df_export['compliance_status'].map({
    'NOT MET': 'P1 — Immediate',
    'PARTIAL': 'P2 — Short-term',
    'UNKNOWN': 'P3 — Verify',
    'MET':     'P4 — Maintain',
})

csv_path = '/home/claude/project/7CS525_compliance_checklist.csv'
df_export.to_csv(csv_path, index=False)
print(f"Compliance checklist exported: {csv_path}")
print(f"Total rows: {len(df_export)}")
print()

# ── Executive Summary ────────────────────────────────────────────────────────
print("=" * 70)
print("  EXECUTIVE COMPLIANCE SUMMARY")
print("=" * 70)
print(f"  System:          {system_profile['system_name']}")
print(f"  Deployer:        {system_profile['deploying_organisation']}")
print(f"  Classification:  {classification['risk_level']} (EU AI Act)")
print(f"  Annex III refs:  {', '.join(classification['annex_refs'])}")
print()
print(f"  Obligations assessed: {len(obligations)}")
print(f"  ✅  Met:               {met}   ({met/len(obligations):.0%})")
print(f"  ⚠️   Partial:           {partial}   ({partial/len(obligations):.0%})")
print(f"  ❌  Not Met:           {not_met}  ({not_met/len(obligations):.0%})")
print(f"  ❓  Unknown:           {unknown}   ({unknown/len(obligations):.0%})")
print()
p1_items = df_export[df_export['priority'] == 'P1 — Immediate']['obligation'].tolist()
print(f"  IMMEDIATE ACTIONS REQUIRED (P1):")
for item in p1_items:
    print(f"    → {item}")
print("=" * 70)
print()
print("Notebook complete. Outputs:")
print("  1. compliance_risk_profile.png")
print("  2. demographic_bias_chart.png")
print("  3. 7CS525_compliance_checklist.csv")

---
## References

1. European Parliament and Council 2024. Regulation (EU) 2024/1689 (Artificial Intelligence Act). *Official Journal of the European Union.* https://artificialintelligenceact.eu/.

2. UK Government 2023. *A pro-innovation approach to AI regulation.* GOV.UK.

3. Buolamwini J. and Gebru T. 2018. Gender Shades: Intersectional Accuracy Disparities in Commercial Gender Classification. *Proceedings of Machine Learning Research* 81, 1–15.

4. NIST 2019. *Face Recognition Vendor Test (FRVT) Part 3: Demographic Effects.* NISTIR 8280.

5. ICO 2022. *Guidance on Biometric Data.* Information Commissioner's Office.

6. *R (Bridges) v Chief Constable of South Wales Police* [2020] EWCA Civ 1058.

---
*7CS525 Assessment 01 — Automation Artefact (Component 5) | University of Derby | 2025/26*